In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 22.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Bidirectional, Dropout
from tensorflow.keras.layers import Layer
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import tensorflow.keras.backend as K
import matplotlib.pyplot as plt

In [ ]:
# Custom Attention Layer
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight',
                                 shape=(input_shape[-1], 1),
                                 initializer='random_normal',
                                 trainable=True)
        self.b = self.add_weight(name='attention_bias',
                                 shape=(input_shape[1], 1),
                                 initializer='zeros',
                                 trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, inputs):
        e = K.tanh(K.dot(inputs, self.W) + self.b)
        e = K.squeeze(e, axis=-1)
        alpha = K.softmax(e)
        alpha = K.expand_dims(alpha, axis=-1)
        context = inputs * alpha
        context = K.sum(context, axis=1)
        return context

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

# Load and merge CSV files
def load_and_merge_data(file_paths):
    dfs = []
    for file in file_paths:
        df = pd.read_csv(file)
        print(f"Columns in {file}: {df.columns.tolist()}")
        dfs.append(df)
    merged_df = pd.concat(dfs, ignore_index=True)
    return merged_df

# Preprocess data
def preprocess_data(df, initial_features, target, time_steps=5, is_train=True, scaler=None):
    df['Date '] = pd.to_datetime(df['Date '], format='%d-%b-%Y')
    df = df.sort_values('Date ')
    if 'Turnover (₹ Cr)' in initial_features:
        df['Turnover (₹ Cr)'] = df['Turnover (₹ Cr)'] * 1e7

    # Calculate 'Previous' day's Close
    df['Previous'] = df['Close '].shift(1)

    # Use ONLY the features from the paper, mapping "Volume" to "Shares Traded"
    features = ['Open ', 'High ', 'Low ', 'Close ', 'Shares Traded ', 'Previous']
    required_cols = features + [target]
    df = df.dropna(subset=required_cols)

    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].isnull().any():
            raise ValueError(f"Non-numeric values found in {col} after dropping missing rows.")

    data = df[features + [target]].values

    if is_train:
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(data)
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test/validation data.")
        scaled_data = scaler.transform(data)

    X, y = [], []
    for i in range(len(scaled_data) - time_steps):
        X.append(scaled_data[i:i + time_steps, :-1])
        y.append(scaled_data[i + time_steps, -1])

    X = np.array(X)
    y = np.array(y)

    return X, y, features, scaler

# Build ABiGRU model
def build_abigru_model(input_shape):
    model = Sequential()
    model.add(Bidirectional(GRU(69, return_sequences=True), input_shape=input_shape))
    model.add(Dropout(0.10270003103822209))
    model.add(Bidirectional(GRU(69, return_sequences=True)))
    model.add(Dropout(0.10270003103822209))
    model.add(Attention())
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.000559481904256915, clipnorm=1.0), loss='mse')
    return model

# Evaluate model
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return mse, rmse, mae

In [ ]:
# Main execution
def main():
    # File paths
    train_files = ['/content/NIFTY FINANCIAL SERVICES-01-01-2022-to-31-12-2022.csv',
                   '/content/NIFTY FINANCIAL SERVICES-01-01-2023-to-31-12-2023.csv',
                   '/content/NIFTY FINANCIAL SERVICES-01-01-2024-to-31-12-2024.csv']
    test_file = '/content/NIFTY FINANCIAL SERVICES-01-01-2025-to-30-04-2025.csv'

    # Initial features and target
    initial_features = ['Open ', 'High ', 'Low ', 'Close ', 'Shares Traded ']  #  Matches CSV column
    target = 'Close '

    # Load and merge data
    train_df = load_and_merge_data(train_files)
    test_df = load_and_merge_data([test_file])

    # Verify required columns
    required_cols = initial_features + [target]
    missing_cols_train = [col for col in required_cols if col not in train_df.columns]
    if missing_cols_train:
        raise KeyError(f"Missing columns in training data: {missing_cols_train}")
    missing_cols_test = [col for col in required_cols if col not in test_df.columns]
    if missing_cols_test:
        raise KeyError(f"Missing columns in test data: {missing_cols_test}")

    # Preprocess training data
    time_steps = 2
    X_train_full, y_train_full, features, train_scaler = preprocess_data(train_df, initial_features, target, time_steps, is_train=True)

    # Split training data
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, shuffle=False)

    # Preprocess test data
    X_test, y_test, _, _ = preprocess_data(test_df, initial_features, target, time_steps, is_train=False, scaler=train_scaler)

    # Build and train model
    model = build_abigru_model(input_shape=(time_steps, len(features)))
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
    early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1)
    history = model.fit(X_train, y_train, epochs=200, batch_size=32,
                        validation_data=(X_val, y_val),
                        callbacks=[lr_scheduler, early_stopping],
                        verbose=1)

    # Predict on test data
    y_pred_scaled = model.predict(X_test)

    # Evaluate model on normalized data
    mse, rmse, mae = evaluate_model(y_test, y_pred_scaled.flatten())
    print(f"Test MSE (normalized): {mse:.4f}")
    print(f"Test RMSE (normalized): {rmse:.4f}")
    print(f"Test MAE (normalized): {mae:.4f}")

    # Inverse transform predictions and true values
    num_columns = len(features) + 1
    dummy_pred = np.zeros((len(y_pred_scaled), num_columns))
    dummy_true = np.zeros((len(y_test), num_columns))
    dummy_pred[:, -1] = y_pred_scaled.flatten()
    dummy_true[:, -1] = y_test.flatten()
    y_pred = train_scaler.inverse_transform(dummy_pred)[:, -1]
    y_true = train_scaler.inverse_transform(dummy_true)[:, -1]

    # Evaluate model on original scale data
    mse_raw, rmse_raw, mae_raw = evaluate_model(y_true, y_pred)
    print(f"\nTest MSE (raw): {mse_raw:.4f}")
    print(f"Test RMSE (raw): {rmse_raw:.4f}")
    print(f"Test MAE (raw): {mae_raw:.4f}")

    # Plot true vs predicted values
    plt.figure(figsize=(12, 6))
    plt.plot(y_true, label='True Close Price', color='green', linewidth=2)
    plt.plot(y_pred, label='Predicted Close Price', color='red', linestyle='--', linewidth=2)
    plt.title('True vs Predicted Close Prices (2025)', fontsize=14)
    plt.xlabel('Time Step', fontsize=12)
    plt.ylabel('Close Price (₹)', fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True)
    plt.savefig('true_vs_predicted_optimized.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Plot training and validation loss curves
    plt.figure(figsize=(12, 6))
    plt.plot(history.history['loss'], label='Training Loss', color='blue', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', color='orange', linewidth=2)
    plt.title('Training and Validation Loss Curves', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
    plt.legend(fontsize=10)
    plt.grid(True)
    plt.savefig('loss_curve.png', dpi=300, bbox_inches='tight')
    plt.close()

if __name__ == "__main__":
    main()

Columns in /content/NIFTY FINANCIAL SERVICES-01-01-2022-to-31-12-2022.csv: ['Date ', 'Open ', 'High ', 'Low ', 'Close ', 'Shares Traded ', 'Turnover (₹ Cr)']
Columns in /content/NIFTY FINANCIAL SERVICES-01-01-2023-to-31-12-2023.csv: ['Date ', 'Open ', 'High ', 'Low ', 'Close ', 'Shares Traded ', 'Turnover (₹ Cr)']
Columns in /content/NIFTY FINANCIAL SERVICES-01-01-2024-to-31-12-2024.csv: ['Date ', 'Open ', 'High ', 'Low ', 'Close ', 'Shares Traded ', 'Turnover (₹ Cr)']
Columns in /content/NIFTY FINANCIAL SERVICES-01-01-2025-to-30-04-2025.csv: ['Date ', 'Open ', 'High ', 'Low ', 'Close ', 'Shares Traded ', 'Turnover (₹ Cr)']
Epoch 1/200


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


19/19 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - loss: 0.0803 - val_loss: 0.0015 - learning_rate: 5.5948e-04
Epoch 2/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0035 - val_loss: 0.0049 - learning_rate: 5.5948e-04
Epoch 3/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0018 - val_loss: 0.0058 - learning_rate: 5.5948e-04
Epoch 4/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0012 - val_loss: 0.0022 - learning_rate: 5.5948e-04
Epoch 5/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 9.9890e-04 - val_loss: 0.0018 - learning_rate: 5.5948e-04
Epoch 6/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0010 - val_loss: 0.0016 - learning_rate: 5.5948e-04
Epoch 7/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0011 - val_loss: 0.0014 - learning_rate: 5.5948e-04
Epoch 8/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 9.5396e-04 - val_loss: 0.0014 - learning_rate: 5.5948e-04
Epoch 9/200
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 8.8787e-04 - val_loss: 0.0013 - 